# sbeam — Analytical Methods

This notebook documents the theoretical basis of the `sbeam` finite element analysis program.
It covers the derivation of the Euler-Bernoulli element stiffness matrix, the consistent mass
matrix, the coordinate transformation, and the eigenvalue solution procedure used in SOL 101
and SOL 103.

**References:**
- NASA-CR-145949
- https://www.sesamx.io/blog/beam_finite_element/
- https://mechanicalc.com/reference/finite-element-analysis

In [ ]:
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt

---
## 1. Euler-Bernoulli Beam Theory

### Assumptions

1. **Plane sections remain plane** — cross-sections perpendicular to the neutral axis before
   deformation remain plane and perpendicular after deformation.
2. **No shear deformation** — transverse shear strain is zero (valid for slender beams, L/d > ~10).
3. **Small displacements** — rotations are small so sin θ ≈ θ.
4. **Linear elastic material** — stress proportional to strain via Young's modulus E.

### Governing Equation

For transverse displacement v(x) along a beam of length L:

$$EI \frac{d^4 v}{dx^4} = q(x)$$

where q(x) is the distributed transverse load per unit length and I is the second moment of
area about the bending axis.

Each CBAR element has **12 degrees of freedom** — 6 per node (Tx, Ty, Tz, Rx, Ry, Rz) at
end A (node 1) and end B (node 2):

$$\mathbf{u}_e = [u_1,\; v_1,\; w_1,\; \theta_{x1},\; \theta_{y1},\; \theta_{z1},\;
                  u_2,\; v_2,\; w_2,\; \theta_{x2},\; \theta_{y2},\; \theta_{z2}]^T$$

---
## 2. Element Stiffness Matrix

### Local Coordinate System

The element local axes are:
- **Axis 1 (x):** along the element centreline from end A to end B.
- **Axis 2 (y):** lies in the plane defined by axis 1 and the orientation vector.
- **Axis 3 (z):** completes the right-hand triad.

### Shape Functions

Axial and torsional displacement use **linear** shape functions:

$$N_1(x) = 1 - \frac{x}{L}, \quad N_2(x) = \frac{x}{L}$$

Transverse displacement (bending) uses **cubic Hermite** shape functions:

$$H_1(x) = 1 - 3\xi^2 + 2\xi^3, \quad
H_2(x) = L(\xi - 2\xi^2 + \xi^3)$$
$$H_3(x) = 3\xi^2 - 2\xi^3, \quad
H_4(x) = L(-\xi^2 + \xi^3), \quad \xi = x/L$$

### 12×12 Local Stiffness Matrix

The local stiffness matrix $[k_e]$ has four decoupled blocks:

**Axial (DOFs 1, 7):**
$$[k_{ax}] = \frac{EA}{L} \begin{bmatrix} 1 & -1 \\ -1 & 1 \end{bmatrix}$$

**Torsion (DOFs 4, 10):**
$$[k_{tor}] = \frac{GJ}{L} \begin{bmatrix} 1 & -1 \\ -1 & 1 \end{bmatrix}$$

**Bending in plane 1–3 (DOFs 3, 5, 9, 11 — Tz and Ry):**
$$[k_{b13}] = \frac{EI_1}{L^3}
\begin{bmatrix}
 12 & -6L & -12 & -6L \\
-6L &  4L^2 & 6L &  2L^2 \\
-12 &  6L &  12 &  6L \\
-6L &  2L^2 & 6L &  4L^2
\end{bmatrix}$$

**Bending in plane 1–2 (DOFs 2, 6, 8, 12 — Ty and Rz):**
$$[k_{b12}] = \frac{EI_2}{L^3}
\begin{bmatrix}
 12 &  6L & -12 &  6L \\
 6L &  4L^2 & -6L &  2L^2 \\
-12 & -6L &  12 & -6L \\
 6L &  2L^2 & -6L &  4L^2
\end{bmatrix}$$

where I₁ and I₂ are moments of inertia about local axes 1 and 2 respectively (PBAR fields).

In [ ]:
def local_stiffness(E, G, A, I1, I2, J, L):
    """12x12 Euler-Bernoulli local element stiffness matrix."""
    k = np.zeros((12, 12))

    # Axial: DOFs 0, 6
    ea_l = E * A / L
    for i, j, v in [(0,0,ea_l),(0,6,-ea_l),(6,0,-ea_l),(6,6,ea_l)]:
        k[i, j] = v

    # Torsion: DOFs 3, 9
    gj_l = G * J / L
    for i, j, v in [(3,3,gj_l),(3,9,-gj_l),(9,3,-gj_l),(9,9,gj_l)]:
        k[i, j] = v

    # Bending in 1-3 plane (Tz, Ry): DOFs 2, 4, 8, 10
    c = E * I1 / L**3
    d = [(2,2,12*c),(2,4,-6*L*c),(2,8,-12*c),(2,10,-6*L*c),
         (4,4,4*L**2*c),(4,8,6*L*c),(4,10,2*L**2*c),
         (8,8,12*c),(8,10,6*L*c),(10,10,4*L**2*c)]
    for i, j, v in d:
        k[i, j] = v; k[j, i] = v

    # Bending in 1-2 plane (Ty, Rz): DOFs 1, 5, 7, 11
    c = E * I2 / L**3
    d = [(1,1,12*c),(1,5,6*L*c),(1,7,-12*c),(1,11,6*L*c),
         (5,5,4*L**2*c),(5,7,-6*L*c),(5,11,2*L**2*c),
         (7,7,12*c),(7,11,-6*L*c),(11,11,4*L**2*c)]
    for i, j, v in d:
        k[i, j] = v; k[j, i] = v

    return k


# Verification: axial-only element
E, G, A, I1, I2, J, L = 2e11, 7.7e10, 0.05, 0.0, 0.0, 0.0, 1.0
k = local_stiffness(E, G, A, I1, I2, J, L)
print("Axial stiffness EA/L =", E*A/L)
print("k[0,0] =", k[0,0], "  k[0,6] =", k[0,6])
print("Symmetric:", np.allclose(k, k.T))
print("All eigenvalues >= 0:", np.all(np.linalg.eigvalsh(k) >= -1e-10))

### Verification — Cantilever Tip Deflection (Single Element)

For a cantilever of length L with tip load P in Y, the exact deflection is:
$$\delta_{tip} = \frac{PL^3}{3EI}$$

A single Euler-Bernoulli element reproduces this **exactly** because the cubic Hermite shape
functions are the exact solution to the governing ODE under a point load.

In [ ]:
E, G, A, I1, I2, J, L, P = 2e11, 7.7e10, 0.05, 8.333e-4, 8.333e-4, 1e-4, 1.0, 1000.0

k = local_stiffness(E, G, A, I1, I2, J, L)

# Constrain DOFs at node A (indices 0-5): all fixed
free = list(range(6, 12))
k_free = k[np.ix_(free, free)]
f_free = np.zeros(6)
f_free[1] = P  # Ty at node B (index 7 → local index 1)

u_free = np.linalg.solve(k_free, f_free)

tip_ty = u_free[1]
exact   = P * L**3 / (3 * E * I2)

print(f"FEA tip Ty  = {tip_ty:.6e} m")
print(f"Exact PL³/3EI = {exact:.6e} m")
print(f"Error = {abs(tip_ty - exact)/exact * 100:.4f}%")

---
## 3. Consistent Mass Matrix

The consistent mass matrix is derived from the same shape functions as the stiffness matrix,
so that element kinetic energy is $T = \frac{1}{2}\dot{\mathbf{u}}^T [m_e] \dot{\mathbf{u}}$.

$$[m_e] = \int_0^L \rho A\; [\mathbf{N}]^T [\mathbf{N}]\; dx$$

### Axial and Torsional Sub-Matrices

Using linear shape functions, the axial translational mass (DOFs 1, 7) and torsional rotational
mass (DOFs 4, 10) take the same form:

$$[m_{ax}] = \frac{\rho A L}{6} \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}, \quad
[m_{tor}] = \frac{\rho I_p L}{6} \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$$

where $I_p = J$ (polar moment). Note: rotational inertia about the element axis is included
but the transverse rotational inertia terms are often small and can be zeroed (Euler-Bernoulli
assumption); `sbeam` retains them from the cubic Hermite integration.

### Bending Sub-Matrix

Using cubic Hermite functions for transverse displacement in the 1–2 plane
(DOFs v₁, θ₁, v₂, θ₂):

$$[m_{b}] = \frac{\rho A L}{420}
\begin{bmatrix}
 156 &  22L &  54 & -13L \\
 22L &  4L^2 &  13L & -3L^2 \\
  54 &  13L & 156 & -22L \\
-13L & -3L^2 & -22L &  4L^2
\end{bmatrix}$$

The same form applies for the 1–3 bending plane. The full 12×12 local mass matrix
assembles these four decoupled blocks.

In [ ]:
def local_mass(rho, A, J, L):
    """12x12 consistent local element mass matrix."""
    m = np.zeros((12, 12))
    ral = rho * A * L

    # Axial translational: DOFs 0, 6
    for (i,j), v in [((0,0),2/6),((0,6),1/6),((6,6),2/6)]:
        m[i,j] = ral*v; m[j,i] = ral*v

    # Torsional rotational: DOFs 3, 9 (Ip ≈ J for circular; use J as placeholder)
    rjl = rho * J * L
    for (i,j), v in [((3,3),2/6),((3,9),1/6),((9,9),2/6)]:
        m[i,j] = rjl*v; m[j,i] = rjl*v

    # Bending 1-3 (Tz, Ry): DOFs 2, 4, 8, 10
    c = ral / 420
    bm = np.array([
        [ 156,  22*L,   54, -13*L],
        [22*L, 4*L**2, 13*L,-3*L**2],
        [  54,  13*L,  156, -22*L],
        [-13*L,-3*L**2,-22*L, 4*L**2]
    ]) * c
    for li, gi in enumerate([2, 4, 8, 10]):
        for lj, gj in enumerate([2, 4, 8, 10]):
            m[gi, gj] += bm[li, lj]

    # Bending 1-2 (Ty, Rz): DOFs 1, 5, 7, 11  (signs flip for Rz)
    bm2 = np.array([
        [ 156, -22*L,   54,  13*L],
        [-22*L, 4*L**2,-13*L,-3*L**2],
        [  54, -13*L,  156,  22*L],
        [ 13*L,-3*L**2, 22*L, 4*L**2]
    ]) * c
    for li, gi in enumerate([1, 5, 7, 11]):
        for lj, gj in enumerate([1, 5, 7, 11]):
            m[gi, gj] += bm2[li, lj]

    return m


# Verification: total mass = rho * A * L
rho, A, J, L = 7850.0, 0.05, 1e-4, 1.0
m = local_mass(rho, A, J, L)
expected_mass = rho * A * L
# Sum of row 0 + row 6 of translational DOF block gives total mass (partition test)
print("Expected total mass:", expected_mass)
print("m[0,0]+m[0,6]+m[6,0]+m[6,6] (axial block sum):",
      m[0,0] + m[0,6] + m[6,0] + m[6,6])  # = rho*A*L
print("Symmetric:", np.allclose(m, m.T))
print("Positive definite (all eigenvalues > 0):",
      np.all(np.linalg.eigvalsh(m) > -1e-12))

---
## 4. Coordinate Transformation

The local element stiffness and mass matrices must be transformed to the global coordinate
system before assembly:

$$[K_e^{global}] = [T]^T [k_e] [T], \quad [M_e^{global}] = [T]^T [m_e] [T]$$

### Direction Cosines

The 3×3 rotation matrix $[\lambda]$ maps local axes to global axes. Its rows are the unit
vectors of the local axes expressed in global coordinates:

1. **Local axis 1** (element x): unit vector $\hat{e}_1 = (\mathbf{x}_B - \mathbf{x}_A)/L$
2. **Local axis 3** (element z): determined from axis 1 and the orientation vector $\mathbf{v}$:
   $$\hat{e}_3 = \frac{\hat{e}_1 \times \mathbf{v}}{|\hat{e}_1 \times \mathbf{v}|}$$
   (Note: sbeam uses the NASTRAN convention — orientation vector defines the 1–2 plane.)
3. **Local axis 2** (element y): completes the right-hand triad:
   $$\hat{e}_2 = \hat{e}_3 \times \hat{e}_1$$

### 12×12 Transformation Matrix

The full transformation matrix $[T]$ is block-diagonal with four copies of $[\lambda]$
(one per nodal DOF triplet — translational and rotational at each end):

$$[T] = \text{blkdiag}([\lambda],\; [\lambda],\; [\lambda],\; [\lambda])$$

This ensures both the translational DOFs (Tx, Ty, Tz) and the rotational DOFs (Rx, Ry, Rz)
at each node transform identically, consistent with the assumption that moments transform
as pseudo-vectors.

In [ ]:
def transform_matrix(xa, xb, v_orient):
    """Build 12x12 transformation matrix T from end-point coords and orientation vector."""
    e1 = np.array(xb) - np.array(xa)
    L = np.linalg.norm(e1)
    e1 = e1 / L

    v = np.array(v_orient, dtype=float)
    e3 = np.cross(e1, v)
    e3 /= np.linalg.norm(e3)
    e2 = np.cross(e3, e1)

    lam = np.array([e1, e2, e3])  # rows = local axes in global coords

    T = np.zeros((12, 12))
    for i in range(4):
        T[3*i:3*i+3, 3*i:3*i+3] = lam
    return T


# Verification: element along global X — T should be identity
xa, xb = [0, 0, 0], [1, 0, 0]
v_orient = [0, 0, 1]  # Z is the orientation reference
T = transform_matrix(xa, xb, v_orient)
print("Element along X, orientation Z — T is identity:", np.allclose(T, np.eye(12)))

# Verification: transformation is orthogonal (T^T T = I)
xa, xb = [0, 0, 0], [1, 1, 0]  # 45° in XY plane
T45 = transform_matrix(xa, xb, v_orient)
print("T is orthogonal:", np.allclose(T45.T @ T45, np.eye(12)))

# Stiffness transformation preserves symmetry and eigenvalues
E, G, A, I1, I2, J = 2e11, 7.7e10, 0.05, 8.333e-4, 8.333e-4, 1e-4
L = np.linalg.norm(np.array(xb) - np.array(xa))
k_local = local_stiffness(E, G, A, I1, I2, J, L)
k_global = T45.T @ k_local @ T45
print("Global K symmetric:", np.allclose(k_global, k_global.T))
print("All global K eigenvalues >= 0:",
      np.all(np.linalg.eigvalsh(k_global) >= -1e-10))

---
## 5. Global Assembly

Each element contributes its 12×12 global stiffness (or mass) matrix to the global
$(6N \times 6N)$ system, where N is the number of grid points.

**DOF index convention** — for grid point `i` (0-based), local DOF `d` (0–5):
$$\text{global index} = 6i + d$$

**Scatter operation:**
```
dofs_A = [6*i_A + d for d in range(6)]
dofs_B = [6*i_B + d for d in range(6)]
dofs   = dofs_A + dofs_B          # length 12
K[np.ix_(dofs, dofs)] += K_e_global
```

### Boundary Condition Application (Elimination)

SPC-constrained DOFs are removed by partitioning:

$$\begin{bmatrix} K_{ff} & K_{fc} \\ K_{cf} & K_{cc} \end{bmatrix}
\begin{bmatrix} u_f \\ 0 \end{bmatrix}
= \begin{bmatrix} f_f \\ f_c \end{bmatrix}$$

The free partition $K_{ff}\, u_f = f_f$ is solved. SPC reaction forces are recovered as:
$$f_{spc} = K\, u_{full} - f_{applied}$$
evaluated at constrained DOF positions.

---
## 6. SOL 101 — Static Solution

The assembled and partitioned system:

$$[K_{ff}]\{u_f\} = \{f_f\}$$

is solved by direct factorisation (`numpy.linalg.solve`, which uses LU decomposition via
LAPACK). This is exact within floating-point precision and appropriate for the phase 1
constraint of ≤ 200 elements (maximum free DOF count ≈ 1200).

### Post-Processing

**CBAR end forces** (local coordinates):
$$\{f_{e,local}\} = [k_e]\,[T]\,\{u_e\}$$

**Stress at PBAR recovery point $(y, z)$:**
$$\sigma = \frac{F_1}{A} + \frac{M_2\, z}{I_1} + \frac{M_3\, y}{I_2}$$

where $F_1$ is the axial force, $M_2$ and $M_3$ are the bending moments about local axes 2
and 3 (from CBAR end forces), and $y, z$ are the recovery point offsets on the PBAR card.

---
## 7. SOL 103 — Generalised Eigenvalue Problem

The undamped free-vibration equation after applying boundary conditions:

$$[K_{ff}]\{\phi\} = \omega^2 [M_{ff}]\{\phi\}$$

Natural circular frequency: $\omega$ (rad/s). Natural frequency: $f = \omega / 2\pi$ (Hz).

### Solver

`scipy.linalg.eigh(K_ff, M_ff)` is used. It solves the symmetric positive-definite
generalised eigenvalue problem and returns eigenvalues in ascending order. This is
equivalent to the Lanczos method for this problem size.

Eigenvalues $\lambda_i = \omega_i^2 \geq 0$ (may be slightly negative due to round-off for
rigid body modes). The implementation clamps: $\omega_i = \sqrt{\max(\lambda_i, 0)}$.

### Mode Normalisation

**MASS normalisation (default, EIGRL NORM=MASS):**
$$\{\hat{\phi}_i\} = \frac{\{\phi_i\}}{\sqrt{\{\phi_i\}^T [M_{ff}] \{\phi_i\}}}$$
Result: $\{\hat{\phi}_i\}^T [M_{ff}] \{\hat{\phi}_i\} = 1$ (unit modal mass).

**MAX normalisation (EIGRL NORM=MAX):**
$$\{\hat{\phi}_i\} = \frac{\{\phi_i\}}{\max|\phi_i|}$$
Result: largest component of each mode shape = 1.0.

### Modal Effective Mass

For translational direction $d$ with unit rigid-body vector $\{T_d\}$:

$$L_{id} = \{\hat{\phi}_i\}^T [M_{ff}] \{T_d\}$$
$$m_{eff,id} = L_{id}^2$$

The sum $\sum_i m_{eff,id}$ should approach total structural mass when sufficient modes
are extracted.

In [ ]:
# Cantilever beam: 10-element FEA vs analytical first bending frequency

n_elem = 10
E, G, rho = 2e11, 7.7e10, 7850.0
A, I, J    = 0.05, 8.333e-4, 1e-4
L_total    = 1.0
L_e        = L_total / n_elem
n_nodes    = n_elem + 1
n_dof      = 6 * n_nodes

K = np.zeros((n_dof, n_dof))
M = np.zeros((n_dof, n_dof))
v_orient = [0, 0, 1]

for e in range(n_elem):
    xa = [e * L_e, 0, 0]
    xb = [(e+1) * L_e, 0, 0]
    T  = transform_matrix(xa, xb, v_orient)
    ke = local_stiffness(E, G, A, I, I, J, L_e)
    me = local_mass(rho, A, J, L_e)
    ke_g = T.T @ ke @ T
    me_g = T.T @ me @ T
    dofs = list(range(6*e, 6*e+12))
    K[np.ix_(dofs, dofs)] += ke_g
    M[np.ix_(dofs, dofs)] += me_g

# Cantilever: fix all 6 DOFs at node 0
fixed = list(range(6))
free  = [i for i in range(n_dof) if i not in fixed]
K_ff  = K[np.ix_(free, free)]
M_ff  = M[np.ix_(free, free)]

eigvals, eigvecs = linalg.eigh(K_ff, M_ff)
omega = np.sqrt(np.maximum(eigvals, 0))
freq  = omega / (2 * np.pi)

# Analytical: f1 = (1.8751^2 / 2pi) * sqrt(EI / rho*A*L^4)
beta1_L = 1.87510407
f1_exact = (beta1_L**2 / (2 * np.pi)) * np.sqrt(E * I / (rho * A * L_total**4))

print(f"FEA f1    = {freq[0]:.4f} Hz")
print(f"Exact f1  = {f1_exact:.4f} Hz")
print(f"Error     = {abs(freq[0] - f1_exact)/f1_exact*100:.4f}%")

---
## 8. RBE3 Constraint — DOF Transformation

An RBE3 element constrains a dependent (reference) grid DOF to be a weighted average of
independent grid DOFs. This is implemented as a DOF-transformation matrix $[T]$ (not to be
confused with the element coordinate transformation) that maps the reduced DOF space back to
the full DOF space.

### Construction

For a dependent DOF $p$ (REFGRID global index) and independent grids with weights $w_i$:

$$u_p = \frac{\sum_i w_i\, u_{q_i}}{\sum_i w_i}$$

The transformation matrix $[T_{full}]$ (shape $n_{dof} \times n_{dof}$) starts as identity.
Row $p$ is replaced by the weighted average over independent DOF columns $q_i$:

$$T_{full}[p, q_i] = \frac{w_i}{W}, \quad W = \sum_i w_i, \quad T_{full}[p, p] = 0$$

Dependent DOF columns are then removed (they are no longer independent variables):

$$[T] = T_{full}[:, \text{red\_dofs}] \quad (\text{shape } n_{dof} \times n_{red})$$

### Application

Before SPC partitioning:
$$[K_{red}] = [T]^T [K] [T], \quad [M_{red}] = [T]^T [M] [T]$$

After solving for reduced displacements $\{u_{red}\}$:
$$\{u_{full}\} = [T]\{u_{red}\}$$

This automatically recovers the dependent DOF displacements as the prescribed weighted
average of the independent DOF solutions.

---
## 9. Convergence Study

Euler-Bernoulli elements with cubic Hermite shape functions reproduce exact static deflections
with a **single element** for polynomial loads up to cubic. For point loads and moment loads,
a single element is exact. Frequency accuracy improves with mesh refinement.

In [ ]:
# Frequency convergence vs number of elements (cantilever first bending mode)
E, G, rho = 2e11, 7.7e10, 7850.0
A, I, J, L_total = 0.05, 8.333e-4, 1e-4, 1.0

beta1_L = 1.87510407
f1_exact = (beta1_L**2 / (2*np.pi)) * np.sqrt(E*I / (rho*A*L_total**4))

n_elems = [1, 2, 3, 5, 10, 20]
errors  = []

for n in n_elems:
    Le = L_total / n
    nn = n + 1
    K = np.zeros((6*nn, 6*nn))
    M = np.zeros((6*nn, 6*nn))
    v = [0, 0, 1]
    for e in range(n):
        xa = [e*Le, 0, 0]; xb = [(e+1)*Le, 0, 0]
        T  = transform_matrix(xa, xb, v)
        ke = local_stiffness(E, G, A, I, I, J, Le)
        me = local_mass(rho, A, J, Le)
        d  = list(range(6*e, 6*e+12))
        K[np.ix_(d,d)] += T.T @ ke @ T
        M[np.ix_(d,d)] += T.T @ me @ T
    free = list(range(6, 6*nn))
    ev, _ = linalg.eigh(K[np.ix_(free,free)], M[np.ix_(free,free)])
    f1 = np.sqrt(max(ev[0], 0)) / (2*np.pi)
    errors.append(abs(f1 - f1_exact)/f1_exact * 100)

fig, ax = plt.subplots()
ax.loglog(n_elems, errors, 'o-')
ax.set_xlabel("Number of elements")
ax.set_ylabel("Error in f₁ (%)")
ax.set_title("Cantilever first bending mode — mesh convergence")
ax.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

for n, e in zip(n_elems, errors):
    print(f"  n={n:3d} elements: error = {e:.4f}%")